# VQA LLMs Competition — Model Run Workflow

This notebook implements the connection to LLMs (Large Language Models) for the **Visual Question Answering (VQA)** task.

**What this script does:**
1. Connects to a local LLM server (LM Studio, Ollama, vLLM, etc.) via an OpenAI-compatible API
2. Sends **multimodal requests** — text + image
3. Receives model answers to VQA questions

**Requirements:**
```
pip install openai pandas
```

---
## 1. Import Libraries and Settings

In [8]:
import json
import base64
import csv
from datetime import datetime
from pathlib import Path
from openai import OpenAI

---
## 2. Connection Configuration

Specify the address of the local LLM server. The OpenAI-compatible API allows using
the same client to connect to:
- **LM Studio** (usually port 1234)
- **Ollama** (usually port 11434)
- **vLLM / Text Generation Inference** (any port)
- **OpenAI API** (requires a real key)

For a local server, the API key is usually not verified, so any value can be specified.

In [61]:
# ── Local Connection ─────────────────────────────────────────────
# Address and port of the local LLM server
LOCAL_API_URL = "http://192.168.31.221:18789/v1"

# No key is required for a local server — specifying a placeholder
API_KEY = "no-api-key"

# Initializing the client that points to the local server
client = OpenAI(
    base_url=LOCAL_API_URL,
    api_key=API_KEY
)

print(f"🔗 Client configured for: {LOCAL_API_URL}")

🔗 Client configured for: http://192.168.31.221:18789/v1


In [63]:
# ── Config: input data-set folder ────────────────────────────────
if Path("data-set").exists():
    DATASET_DIR = Path("data-set")
elif Path("VQA_LLMs_competition/data-set").exists():
    DATASET_DIR = Path("VQA_LLMs_competition/data-set")
else:
    raise FileNotFoundError("❌ Could not find 'data-set' or 'VQA_LLMs_competition/data-set' directory. Please check your working directory.")

# the csv file name and structure
# CSV columns: (index), fname, question, complexity, answer, prompt, negative_prompt, rejected_answers
CSV_FILE = DATASET_DIR / "generations.csv"

# the folder with images (1000 jpg files: 0.jpg – 999.jpg)
IMAGES_DIR = DATASET_DIR / "images"

# ── Verify dataset paths ────────────────────────────────────────
assert DATASET_DIR.exists(), f"❌ Dataset folder not found: {DATASET_DIR}"
assert CSV_FILE.exists(),    f"❌ CSV file not found: {CSV_FILE}"
assert IMAGES_DIR.exists(),  f"❌ Images folder not found: {IMAGES_DIR}"

import pandas as pd
df = pd.read_csv(CSV_FILE)

num_images = len(list(IMAGES_DIR.glob("*.jpg")))
print(f"✅ Dataset loaded from: {DATASET_DIR.resolve()}")
print(f"   📄 CSV file:   {CSV_FILE.name}  ({len(df)} rows)")
print(f"   📁 Images dir:  {IMAGES_DIR.name}  ({num_images} images)")
print(f"   📋 CSV columns: {list(df.columns)}")
print(f"   🔢 Complexity range: {df['complexity'].min()} – {df['complexity'].max()}")


✅ Dataset loaded from: /Users/macuser/GitHub/LoRA-project/VQA_LLMs_competition/data-set
   📄 CSV file:   generations.csv  (1000 rows)
   📁 Images dir:  images  (1000 images)
   📋 CSV columns: ['Unnamed: 0', 'fname', 'question', 'complexity', 'answer', 'prompt', 'negative_prompt', 'rejected_answers']
   🔢 Complexity range: 1 – 4


---
## 3. Connection Verification and Model Selection

Verifying if the server is available and retrieving the list of loaded models.

In [66]:
def check_connection(client: OpenAI) -> list[str]:
    """
    Verifies connection to the LLM server.
    Returns a list of available models or an empty list in case of error.
    """
    try:
        models = client.models.list()
        model_ids = [m.id for m in models.data]
        print(f"✅ Connection successful: {client.base_url}")
        print(f"📦 Available models: {model_ids}")
        return model_ids
    except Exception as e:
        print(f"❌ Connection error: {e}")
        return []


available_models = check_connection(client)

# Select the first available model, or specify manually
MODEL_NAME = available_models[0] if available_models else "local-model"
print(f"\n🚀 Selected model: {MODEL_NAME}")

✅ Connection successful: http://192.168.31.221:18789/v1/
📦 Available models: ['qwen2.5-vl-7b-instruct', 'qwen3.5-27b-claude-4.6-opus-distilled-mlx', 'gemma-4-e2b-heretic-uncensored-mlx', 'gemma-4-e4b-it-mlx', 'text-embedding-nomic-embed-text-v1.5']

🚀 Selected model: qwen2.5-vl-7b-instruct


---
## 4. Helper Functions for Image Processing

There are two ways to send an image to an LLM via the OpenAI API:
- **URL** — pass a link to the image (http/https)
- **Base64** — encode a local file to base64 and pass as `data:image/...;base64,...`

The `prepare_image_content` function automatically detects the type (URL or file) and formats the corresponding block for the API.

In [69]:
def encode_image_to_base64(image_path: str) -> str:
    """
    Reads an image from disk and encodes it into a base64 string.
    Supports formats: png, jpg, jpeg, gif, webp.
    """
    path = Path(image_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {image_path}")

    # Determine MIME type by file extension
    suffix_to_mime = {
        ".png": "image/png",
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".gif": "image/gif",
        ".webp": "image/webp",
    }
    mime = suffix_to_mime.get(path.suffix.lower(), "image/png")

    with open(path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    return f"data:{mime};base64,{encoded}"


def prepare_image_content(image_source: str) -> dict:
    """
    Formats the image_url block for the OpenAI Chat API.

    Parameters:
        image_source: Image URL (http/https) or path to a local file.

    Returns:
        dict in the format {"type": "image_url", "image_url": {"url": ...}}
    """
    if image_source.startswith(("http://", "https://")):
        # Image by URL — pass the link directly
        url = image_source
    else:
        # Local file — encode to base64
        url = encode_image_to_base64(image_source)

    return {
        "type": "image_url",
        "image_url": {"url": url}
    }


print("✅ Helper functions for images loaded")

✅ Helper functions for images loaded


---
## 5. Main LLM Query Function

The `ask_llm` function accepts a text question and (optionally) an image.
It formats a multimodal request and sends it to the model.

**OpenAI Vision API Message Format:**
```json
{
  "role": "user",
  "content": [
    {"type": "text", "text": "What is in this image?"},
    {"type": "image_url", "image_url": {"url": "https://..."}}
  ]
}
```

In [72]:
def ask_llm(
    question: str,
    image_source: str | None = None,
    system_prompt: str = """
    You are a helpful assistant that answers questions about images that you get from request. 
    FIRST OF ALL YOU NEED TO DO IS TO parse AND recognize the image! 
    And first of all, parse the context of the image!
    You answer should be on JSON object in format like 
    {
        question:the question that you are get, 
        answer:["one word per element"]the list of logic SHORT answer of entity that you undestend as good answer,
        reasoning: your full Chain-of-Thought (CoT) how you decide
        
    }
    """,
    model: str | None = None,
    temperature: float = 0.2,
    max_tokens: int = 512,
) -> str:
    """
    Sends a request to the LLM with text and (optionally) an image.

    Parameters:
        question:      Text question for the model.
        image_source:  Image URL or path to a local file. None — text only.
        system_prompt: System prompt (instruction for the model).
        model:         Model name (if None — MODEL_NAME is used).
        temperature:   Generation temperature (0 = deterministic, 1 = creative).
        max_tokens:    Maximum number of tokens in the response.

    Returns:
        Model response text.
    """
    model = model or MODEL_NAME

    # ── Format message content (content) ───────────────────────────
    # Text part is always present
    content = [{"type": "text", "text": question}]

    # If an image is specified — add it to the request
    if image_source is not None:
        content.append(prepare_image_content(image_source))

    # ── Format message list ──────────────────────────────────────────
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": content},
    ]

    # ── Send request ─────────────────────────────────────────────────
    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return response.choices[0].message.content

    except Exception as e:
        return f"❌ API Error: {e}"


print("✅ Function ask_llm is ready to use")

✅ Function ask_llm is ready to use


---
## 7. Test Request — Text + Image (URL)

Sending an image via URL and asking a question about it.

> **Note:** The model must support vision (multimodality).
> For example: LLaVA, Gemma with vision, Qwen-VL, InternVL, etc.

In [75]:
# ── Example: question about image by URL ───────────────────────────
# Can be replaced with any other image link
test_image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/8/89/Tomato_je.jpg/800px-Tomato_je.jpg"

response_vision = ask_llm(
    question="What do you see in this image? Describe the objects.",
    image_source=test_image_url
)

print(f"🖼️  Image: {test_image_url}")
print(f"❓ Question: What do you see in this image?")
print("-" * 50)
print(f"💬 Response: {response_vision}")

🖼️  Image: https://upload.wikimedia.org/wikipedia/commons/thumb/8/89/Tomato_je.jpg/800px-Tomato_je.jpg
❓ Question: What do you see in this image?
--------------------------------------------------
💬 Response: ❌ API Error: Error code: 400 - {'error': "'url' field must be a base64 encoded image."}


---
## 8. Test Request — Text + Local Image (File)

If the image is located on the disk, the function will automatically encode it into base64.

In [78]:
# ── Example: question about a local image ───────────────────────
# Replace the path with a real file from your VQA dataset
local_image_path = "images/test.jpg"  # ← specify real path

if Path(local_image_path).exists():
    response_local = ask_llm(
        question="What objects can you identify in this kitchen scene?",
        image_source=local_image_path
    )
    print(f"🖼️  File: {local_image_path}")
    print("-" * 50)
    print(f"💬 Response: {response_local}")
else:
    print(f"⚠️  File not found: {local_image_path}")
    print("   Replace 'local_image_path' with the actual path to the image.")

🖼️  File: images/test.jpg
--------------------------------------------------
💬 Response: ```json
{
    "question": "What objects can you identify in this kitchen scene?",
    "answer": ["broccoli", "plum", "fillet", "chopping_board", "bread", "pan", "capsicum"],
    "reasoning": "The image depicts a kitchen scene with various food items and utensils. The objects identified include broccoli, plums (which are not typically considered vegetables but are present), a fillet of meat on a chopping board, bread, a frying pan with capsicum (green bell pepper), and the chopping board itself."
}
```


---
## 9. Saving Trial Results to CSV

We implement a function to save the details of each LLM trial (question, image, response, and metadata)
into a CSV file. The file is saved inside the `TestsData` directory with the naming convention `{LLMName}_{date}.csv`.

In [80]:
def save_trial_to_csv(model_name: str, question: str, image_source: str | None, response: str, output_dir: str = "TestsData") -> Path:
    """
    Saves the details of an LLM trial run into a CSV file in the specified directory.
    File name format: {model_name}_{date}.csv
    """
    # Ensure output directory exists
    dir_path = Path(output_dir)
    dir_path.mkdir(parents=True, exist_ok=True)

    # Clean the model name for safe file naming
    clean_model_name = model_name.replace("/", "_").replace("\\", "_")
    
    # Current date and time for the filename and data record
    now = datetime.now()
    filename_date = now.strftime("%Y%m%d_%H%M%S")
    record_date = now.strftime("%Y-%m-%d %H:%M:%S")
    
    file_path = dir_path / f"{clean_model_name}_{filename_date}.csv"
    
    # Define CSV fields
    fieldnames = ["timestamp", "model_name", "question", "image_source", "response"]
    
    with open(file_path, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerow({
            "timestamp": record_date,
            "model_name": model_name,
            "question": question,
            "image_source": image_source or "None",
            "response": response
        })
        
    print(f"💾 Trial results saved to: {file_path}")
    return file_path


# ── Example of saving a trial ──────────────────────────────────────
# save_trial_to_csv(MODEL_NAME, "What is in this image?", local_image_path, response_local)

---
## 10. Connection via API Key (OpenAI, Claude, etc.)

To connect to a cloud API (instead of a local server), you need to:
1. Specify a real API key
2. Change `base_url` to the provider's address

We store keys in `env.json` to avoid hardcoding them in the notebook.

In [82]:
# ── Loading keys from env.json ─────────────────────────────────
ENV_PATH = Path("env.json")

if ENV_PATH.exists():
    with open(ENV_PATH, "r") as f:
        env_config = json.load(f)
    print("✅ Configuration env.json loaded")
    print(f"   Available providers: {list(env_config.keys())}")
else:
    env_config = {}
    print("⚠️  File env.json not found — using local connection only")

✅ Configuration env.json loaded
   Available providers: ['chatGPT', 'claud']


In [83]:
def create_api_client(provider: str, env: dict) -> OpenAI | None:
    """
    Creates an OpenAI client for a cloud provider based on env.json.

    Supported providers and their base_url:
      - chatGPT  → https://api.openai.com/v1
      - claud    → https://api.anthropic.com/v1  (via OpenAI-compatible proxy)

    Parameters:
        provider: Provider name from env.json ("chatGPT", "claud").
        env:      Configuration dictionary (loaded from env.json).

    Returns:
        Configured OpenAI client or None if key is empty/missing.
    """
    if provider not in env:
        print(f"❌ Provider '{provider}' not found in env.json")
        return None

    config = env[provider]
    key = config.get("key", "")

    if not key:
        print(f"⚠️  API key for '{provider}' is empty — please fill env.json")
        return None

    # Determine base_url: if specified in env.json — use it,
    # otherwise — default provider URL
    default_urls = {
        "chatGPT": "https://api.openai.com/v1",
        "claud": "https://api.anthropic.com/v1",
    }
    api_url = config.get("api") or default_urls.get(provider, "")

    if not api_url:
        print(f"❌ Failed to determine API URL for '{provider}'")
        return None

    api_client = OpenAI(base_url=api_url, api_key=key)
    print(f"✅ Client '{provider}' created → {api_url}")
    return api_client


# ── Example of creating a client for ChatGPT ───────────────────────────
# Uncomment once you have populated the key in env.json:
# openai_client = create_api_client("chatGPT", env_config)
# if openai_client:
#     response = ask_llm(
#         question="Describe this image",
#         image_source=test_image_url,
#         model="gpt-4o-mini"
#     )
#     print(response)

print("✅ Function create_api_client ready")

✅ Function create_api_client ready


In [84]:
import csv
from pathlib import Path
from datetime import datetime
from tqdm.notebook import tqdm  # Для красивого відображення прогресу в Jupyter


import json
import re

def parse_json_response(response_str: str) -> dict | None:
    """
    Витягує JSON-об'єкт із тексту відповіді моделі (з підтримкою markdown-блоків)
    і конвертує його в Python-словник.
    """
    if not response_str or not isinstance(response_str, str):
        return None
    
    # 1. Шукаємо вміст усередині ```json ... ``` або ``` ... ```
    match = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', response_str)
    if match:
        json_str = match.group(1).strip()
    else:
        # Якщо маркдаун-блоків немає, шукаємо просто перший підходящий { ... }
        brace_match = re.search(r'\{[\s\S]*\}', response_str)
        json_str = brace_match.group(0).strip() if brace_match else response_str.strip()
        
    # 2. Декодуємо рядок у JSON
    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        print(f"⚠️ Не вдалося розпарсити JSON у тексті: {response_str}")
        return None


# 1. Перевірка з'єднання та вибір моделі
available_models = check_connection(client)
if not available_models:
    raise ConnectionError("❌ З'єднання з LLM сервером відсутнє. Запустіть сервер перед виконанням.")

model_to_use = MODEL_NAME
print(f"🚀 Запуск обробки датасету за допомогою моделі: {model_to_use}")

# 2. Створення папки та вихідного файлу результатів
output_dir = Path("LLMsResults")
output_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
clean_model_name = model_to_use.replace("/", "_").replace("\\", "_")
output_file = output_dir / f"{clean_model_name}_results_{timestamp}.csv"

# Назви колонок для підсумкового файлу
fieldnames = ["index", "image_path", "question", "complexity", "ground_truth", "llm_response", "timestamp"]

# Створюємо файл та записуємо заголовок (header)
with open(output_file, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

print(f"📄 Створено файл для результатів: {output_file}")
print(f"⏳ Починаємо надсилання запитів до LLM ({len(df)} рядків)...")

# 3. Основний цикл обробки даних
# Використовуємо tqdm для інтерактивного прогрес-бару в Jupyter Notebook
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Обробка VQA запитів"):
    # Динамічно визначаємо шлях до зображення
    image_path = DATASET_DIR / row['fname']
    image_source = str(image_path) if image_path.exists() else None
    
    if image_source is None:
        print(f"⚠️ Зображення не знайдено: {row['fname']} (рядок {idx}). Надсилаємо лише текст.")
        
    # 4. Запит до моделі через існуючу функцію
    response = ask_llm(
        question=row['question'],
        image_source=image_source,
        model=model_to_use,
        temperature=0.2,   # Низька температура для більш точних відповідей
        max_tokens=256     # Обмежуємо довжину відповіді для швидкості
    )
    
    # 5. Інкрементне збереження (дозапис) результату в CSV-файл
    with open(output_file, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow({
            "index": idx,
            "image_path": row['fname'],
            "question": row['question'],
            "complexity": row['complexity'],
            "ground_truth": row['answer'],
            "llm_response": response,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

print(f"\n🎉 Робочий процес успішно завершено!")
print(f"💾 Усі результати збережено в: {output_file}")


✅ Connection successful: http://192.168.31.221:18789/v1/
📦 Available models: ['qwen2.5-vl-7b-instruct', 'qwen3.5-27b-claude-4.6-opus-distilled-mlx', 'gemma-4-e2b-heretic-uncensored-mlx', 'gemma-4-e4b-it-mlx', 'text-embedding-nomic-embed-text-v1.5']
🚀 Запуск обробки датасету за допомогою моделі: qwen2.5-vl-7b-instruct
📄 Створено файл для результатів: LLMsResults/qwen2.5-vl-7b-instruct_results_20260607_170222.csv
⏳ Починаємо надсилання запитів до LLM (1000 рядків)...


Обробка VQA запитів:   0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 

---
## Summary

| Component | Description |
|-----------|-------------|
| `client` | OpenAI client connected to the local server |
| `ask_llm()` | Main function — sends text + image to the LLM |
| `prepare_image_content()` | Prepares the image (URL or base64) |
| `save_trial_to_csv()` | Saves trial details to `{LLMName}_{date}.csv` inside `TestsData` |
| `create_api_client()` | Creates a client for cloud API (ChatGPT, Claude) |
| `env.json` | Stores API keys for cloud providers |